# Paper results

`./leonardo/save.sh` on the cluster saves the results in `results/data/`; run `git pull`, then run all cells. Tables go to `results/tables/`, figures to `results/figures/`.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
from agentic_quant import paper_results as pr

ROOT = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd() / "results"
DATA = ROOT / "data"
TABLES = ROOT / "tables"
FIGURES = ROOT / "figures"
TABLES.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

records = pr.load(DATA)
print(f"{len(records)} records, models: {[pr.MODEL_LABELS.get(m, m) for m in pr.models_in(records)]}")
pd.DataFrame(records, columns=["_model", "scenario", "condition", "architecture"]).value_counts(["_model", "scenario"]).unstack(fill_value=0) if records else None

## Table: Peer-Preservation Rate (RQ1 Base columns, RQ2 pattern columns)

In [ ]:
matrix = pr.pp_matrix(records)
display(pd.DataFrame({pr.MODEL_LABELS.get(m, m): {k: None if v is None else round(100 * v, 1) for k, v in values.items()} for m, values in matrix.items()}).T.sort_index(axis=1, key=lambda keys: keys.map(str)))
latex = pr.pp_matrix_latex(records)
(TABLES / "pp_matrix.tex").write_text(latex)
print(latex)

## Table: Task Success Rate (RQ2)

In [ ]:
tsr = pr.task_success(records)
display(pd.DataFrame({pr.ARCHITECTURE_SHORT[a]: {f"Task {t}": None if tsr.get((t, a)) is None else round(100 * tsr[(t, a)], 1) for t in pr.TASKS} for a in pr.ARCHITECTURE_ORDER}).T)
latex = pr.task_success_latex(records)
(TABLES / "task_success.tex").write_text(latex)
print(latex)

## Figure: critic outcomes through each pattern (RQ2)

In [ ]:
fig = pr.plot_flows(records, FIGURES / "flows.pdf")

## Table: Latency and Token Overhead (RQ3)

In [ ]:
cost = pr.overhead(records)
display(pd.DataFrame({pr.ARCHITECTURE_SHORT[a]: {**{f"Latency T{t} (s)": cost.get((t, a), (None, None))[0] for t in pr.TASKS}, **{f"Tokens T{t} (%)": None if cost.get((t, a), (None, None))[1] is None else round(100 * cost[(t, a)][1], 1) for t in pr.TASKS}} for a in pr.PATTERNS}).T)
latex = pr.overhead_latex(records)
(TABLES / "overhead.tex").write_text(latex)
print(latex)

## Figure: overhead vs task outcomes (RQ3)

In [ ]:
fig = pr.plot_tradeoff(records, FIGURES / "tradeoff.pdf")
display(pd.DataFrame(pr.tradeoff(records)))